In [2]:
# aggregations
# count(*) counts all rows
# count(distinct col) count unique values
# sum(col) total of a column
# avg(col) mean of a column
# min(col) smallest value
# max(col) largest value

In [3]:
import pandas as pd
import numpy as np
%load_ext sql
%sql duckdb:///:memory:

np.random.seed(22)
suppliers = ["Acme", "GlobalCo", "FastParts", "PrimeMg", "EastCoast"]
categories = ["Electronics", "Hardware", "Consumables", "Apparel"]
warehouses = ["East", "West", "Central"]

rows = []
for i in range(1, 201):
    rows.append({
        "po_id":          f"PO-{str(i).zfill(4)}",
        "supplier":       np.random.choice(suppliers),
        "category":       np.random.choice(categories),
        "warehouse":      np.random.choice(warehouses),
        "order_date":     pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(np.random.randint(0, 365))),
        "amount":         round(np.random.uniform(500, 50000), 2),
        "units":          np.random.randint(10, 500),
        "lead_time_days": np.random.randint(3, 45),
        "on_time":        np.random.choice([True, False], p=[0.8, 0.2])
    })

po = pd.DataFrame(rows)
%sql --persist po
print(f"po: {po.shape}")

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

Success! Persisted po to the database.

po: (200, 9)


In [4]:
%%sql
-- write a query that returns one row per supplier showing all six core aggregations
-- one row per supplier
-- show: supplier, po_count, unique_categories,
--       total_spend, avg_amount, min_amount, max_amount
-- round all decimals to 2 places
-- sort by total_spend descending

select
    supplier,
    count(*) as po_count,
    count(distinct category) as unique_categories,
    round(sum(amount), 2) as total_spend,
    round(avg(amount), 2) as avg_amount,
    round(min(amount), 2) as min_amount,
    round(max(amount), 2) as max_amount
from
    po
group by
    supplier
order by
    total_spend desc


Running query in 'duckdb:///:memory:'

supplier,po_count,unique_categories,total_spend,avg_amount,min_amount,max_amount
EastCoast,54,4,1200154.42,22225.08,1275.07,49202.14
PrimeMg,43,4,1071116.69,24909.69,772.38,47855.66
FastParts,42,4,995411.84,23700.28,2772.82,47326.18
Acme,32,4,819898.26,25621.82,1958.56,49123.34
GlobalCo,29,4,752094.86,25934.31,772.86,47711.66


In [5]:
%%sql

-- S3.2 AGGREGATION WITH WHERE
-- LEFT table: po
-- filter: only Electronics and Hardware
-- group by: warehouse
-- show: warehouse, po_count, total_spend, avg_lead_time
-- sort: total_spend descending

select
    warehouse,
    count(*) as po_count,
    round(sum(amount), 2) as total_spend,
    round(avg(lead_time_days), 2) as avg_lead_time
from
    po
where
    category in ('Electronics', 'Hardware')
group by
    warehouse
order by
    total_spend desc


Running query in 'duckdb:///:memory:'

warehouse,po_count,total_spend,avg_lead_time
East,37,1069314.82,23.62
Central,35,803270.23,26.23
West,29,647251.76,23.34


In [6]:
%%sql

-- S3.3 HAVING WITH MULTIPLE CONDITIONS
-- group by supplier
-- show: supplier, po_count, avg_amount, on_time_count
-- HAVING: po_count > 35 AND avg_amount > 25000 AND on_time_count >= 1

-- note the conditions are strict so no supplier meets this queries HAVING criteria
-- the result is valid

select
    supplier,
    count(*) as po_count,
    round(avg(amount), 2) as avg_amount,
    sum(
        case when
            on_time = true
        then
            1
        else
            0
        end
        ) 
        as on_time_count
from
    po
group by
    supplier
having
    po_count > 35
and
    avg_amount > 25000
and
    on_time_count >= 1
order by
    avg_amount desc


Running query in 'duckdb:///:memory:'

supplier,po_count,avg_amount,on_time_count


In [ ]:
%%sql
-- S3.4 CONDITIONAL AGGREGATION — OTD RATE
-- group by supplier
-- show: supplier, po_count, on_time_count, late_count, otd_rate
-- sort: otd_rate descending
-- note: when multiplying use 100.0 to force floating point division and eliminate truncation

select
    supplier,
    count(*) as po_count,
    sum(
        case when
            on_time = True
        then
            1
        else
            0
        end
    ) as on_time_count,
    sum(
        case when
            on_time = False
        then
            1
        else
            0
        end
    ) as late_count,
    round(sum(
        case when
            on_time = True
        then
            1
        else
            0
        end
    )
    * 100.0
    /
    count(*), 1) as otd_rate
from
    po
group by
    supplier
order by
    otd_rate desc



Running query in 'duckdb:///:memory:'

supplier,po_count,on_time_count,late_count,otd_rate
PrimeMg,43,37,6,86.0
GlobalCo,29,24,5,82.8
Acme,32,26,6,81.3
FastParts,42,34,8,81.0
EastCoast,54,40,14,74.1


In [8]:
po.head(5)

,po_id,supplier,category,warehouse,order_date,amount,units,lead_time_days,on_time
0,PO-0001,EastCoast,Electronics,East,2024-12-22,32055.22,94,11,True
1,PO-0002,FastParts,Consumables,East,2024-02-15,23470.66,103,42,True
2,PO-0003,Acme,Electronics,West,2024-01-28,22382.64,39,21,True
3,PO-0004,GlobalCo,Apparel,Central,2024-01-08,47315.60,33,26,True
4,PO-0005,EastCoast,Apparel,East,2024-12-25,37347.63,123,4,False
